# FedCardioTwin — end-to-end on a single T4

**Continually Updated, Uncertainty-Aware Federated Digital Twins for Patient-Specific Cardiac Monitoring Across Heterogeneous Hospitals** (IEEE J-BHI special issue: Federated Learning and Digital Twins for Smart Healthcare).

Pipeline: download public data → build caches → smoke test → FAST sanity run → FULL experiments → paper tables.

Runtimes on one T4: cache build ~1–2 h (one-time), FAST preset ~15 min, FULL federated suite = overnight-class (split stages across sessions on Kaggle's 12 h limit; every stage is resumable because caches and checkpoints persist).

In [ ]:
# 1. Environment + repo bootstrap
import os, sys, subprocess

REPO_URL = os.environ.get('FCT_REPO_URL', '')  # set to your GitHub URL once pushed

if os.path.exists('fedcardiotwin'):
    REPO_ROOT = os.path.abspath('.')
elif os.path.exists('../fedcardiotwin'):
    REPO_ROOT = os.path.abspath('..')
elif REPO_URL:
    subprocess.run(['git', 'clone', REPO_URL, 'FedCardioTwin'], check=True)
    REPO_ROOT = os.path.abspath('FedCardioTwin')
else:
    raise RuntimeError('Run from inside the repo, or set FCT_REPO_URL.')

os.chdir(REPO_ROOT)
sys.path.insert(0, REPO_ROOT)
!pip install -q -r requirements.txt

# official CinC-2021 evaluation repo (label definitions)
if not os.path.exists('external/evaluation-2021'):
    !git clone -q --depth 1 https://github.com/physionetchallenges/evaluation-2021 external/evaluation-2021

import torch
print('CUDA:', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else '(CPU)')

In [ ]:
# 2. Smoke test — verifies every code path on synthetic data (no download)
!python tests/smoke_test.py

In [ ]:
# 3. Download the public datasets (one-time, ~15 GB raw)
# Track A: CinC-2021 training sources (6 hospital clients)
SOURCES = ['cpsc_2018', 'cpsc_2018_extra', 'georgia', 'chapman_shaoxing', 'ningbo', 'ptb-xl']
BASE = 'https://physionet.org/files/challenge-2021/1.0.3/training'
for s in SOURCES:
    !wget -r -N -c -np -q --show-progress -P data/raw/cinc_dl {BASE}/{s}/

# Track B: original PTB-XL (patient ids + recording dates for the twin loop)
!wget -r -N -c -np -q --show-progress -P data/raw/ptbxl_dl https://physionet.org/files/ptb-xl/1.0.3/

CINC_RAW = 'data/raw/cinc_dl/physionet.org/files/challenge-2021/1.0.3/training'
PTBXL_RAW = 'data/raw/ptbxl_dl/physionet.org/files/ptb-xl/1.0.3'
print('CinC sources:', os.listdir(CINC_RAW))
print('PTB-XL files:', len(os.listdir(PTBXL_RAW)))

In [ ]:
# 4. Build preprocessed caches (one-time, ~1-2 h; float16 memmaps ~2.5 GB)
!python scripts/build_cache.py --track a --raw-dir {CINC_RAW}
!python scripts/build_cache.py --track b --ptbxl-dir {PTBXL_RAW}
!du -sh data/cache/*

In [ ]:
# 5. FAST sanity run (~15 min): confirms learning before committing GPU hours
!python scripts/run_experiments.py --stage centralized --preset fast
!python scripts/run_experiments.py --stage federated --preset fast --strategies fedavg fedbn

In [ ]:
# 6. FULL experiments — the paper runs. Split across sessions as needed.
# 6a. Pooled upper bound (3 seeds)
!python scripts/run_experiments.py --stage centralized --preset full

In [ ]:
# 6b. Main federated comparison (7 strategies x 3 seeds) — the big one.
# Run strategy-by-strategy if you need to fit Kaggle's 12 h sessions:
for strat in ['local', 'fedavg', 'fedprox', 'fedbn', 'fedavgm', 'ditto', 'fedper']:
    !python scripts/run_experiments.py --stage federated --preset full --strategies {strat}

In [ ]:
# 6c. Twin loop (Track B) + 6d. Conformal layer + 6e. Leave-one-hospital-out
!python scripts/run_experiments.py --stage twin --preset full
!python scripts/run_experiments.py --stage conformal
!python scripts/run_experiments.py --stage loho --preset full

In [ ]:
# 6e. Seed ensemble — the headline number (averages the 3 seed checkpoints)
!python scripts/ensemble_eval.py --ckpts checkpoints/central_seed0.pt \
    checkpoints/central_seed1.pt checkpoints/central_seed2.pt

In [ ]:
## Output → paper mapping

| Result file | Paper artifact |
|---|---|
| `results/fed_*_seed*.json` | Table 2: main comparison (per-client, MEAN, WORST, comm MB/round) + `history` field → Fig: AUROC vs rounds / vs cumulative upload MB |
| `results/centralized_seed*.json` | Table 2 top row: pooled upper bound |
| `results/ensemble.json` | Table 2 headline row: 3-seed ensemble of the SE-Inception model |
| `results/loho_seed*.json` | Table: leave-one-hospital-out generalization to an unseen institution |
| `results/twin_seed*.json` | Table 3: cold vs warm AUROC, per-step gain curve, alert-detection AUROC, backward transfer |
| `results/twin_seed*_arrays.npz` | raw per-record arrays → post-hoc analyses (e.g. conformal coverage across twin updates) |
| `results/conformal.json` | Table 4: per-hospital FNR coverage + mean set size (local vs federated λ), computed on deployed per-client models |
| `make_tables.py` output | mean ± std + Wilcoxon signed-rank significance vs FedAvg |

Ablations for Table 5 — each best-results component toggles independently: `se=False` (models), `augment=False`, `mixup_alpha=0`, `ema_decay=0` (configs.py), `--strategies fedavg` vs `fedbn` (hospital tier), `update_steps=0` / `replay_size=1` (twin).

## Output → paper mapping

| Result file | Paper artifact |
|---|---|
| `results/fed_*_seed*.json` | Table 2: main comparison (per-client, MEAN, WORST, comm MB/round) |
| `results/centralized_seed*.json` | Table 2 top row: pooled upper bound |
| `results/twin_seed*.json` | Table 3: cold vs warm AUROC, personalization gain, alert score, backward transfer |
| `results/conformal.json` | Table 4: per-hospital FNR coverage + mean set size (local vs federated λ) |

Ablations = re-run `--stage federated` with `--strategies fedavg` (no hospital tier) vs `fedbn` (with), and `--stage twin` with `update_steps=0` / `replay_size=1` in `fedcardiotwin/configs.py`.